# Lab · Hồi quy nhiều đầu vào và nhiều đầu ra · LỜI GIẢI

> Chỉ mở file này khi bạn đã thử bài tập, hoặc đã dùng hết cả 3 gợi ý.

Bài tập cho [`lesson.md`](../lesson.md) (Module 5). Gom bốn lab của khóa: **A** dự đoán nhiều đầu vào, **B** huấn luyện nhiều đầu vào, **C** dự đoán nhiều đầu ra, **D** huấn luyện nhiều đầu ra.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.
Mẹo: trước mỗi bài, **viết ra shape** của từng tensor.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch", torch.__version__)

## Bài 1. A · Dự đoán nhiều đầu vào bằng ma trận

Tạo `w = [[2.0], [3.0]]` (shape `(2, 1)`) và `b = [[1.0]]`, đều có theo dõi gradient. Viết `forward(x)` dùng `torch.mm(x, w) + b`.
- `yhat_single`: dự đoán cho một mẫu `x = [[1.0, 2.0]]`. Tính tay trước: `1·2 + 2·3 + 1`.
- `yhat_batch`: dự đoán cho `X = [[1., 1.], [1., 2.], [1., 3.]]` (3 mẫu, 2 đặc trưng).
- `shape_batch`: shape của `yhat_batch`.

In [ ]:
w = torch.tensor([[2.0], [3.0]], requires_grad=True)
b = torch.tensor([[1.0]], requires_grad=True)

def forward(x):
    return torch.mm(x, w) + b

X = torch.tensor([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]])
yhat_single = forward(torch.tensor([[1.0, 2.0]]))
yhat_batch = forward(X)
shape_batch = yhat_batch.shape
print(yhat_single, yhat_batch, shape_batch)

In [ ]:
assert yhat_single.tolist() == [[9.0]]
assert yhat_batch.tolist() == [[6.0], [9.0], [12.0]]
assert tuple(shape_batch) == (3, 1)
print("✅ Bài 1 đúng")

## Bài 2. A · `nn.Linear(2, 1)` và shape của tham số

Đặt `torch.manual_seed(1)`, tạo `model = nn.Linear(2, 1)`. Lưu:
- `w_shape`, `b_shape`: shape của `model.weight` và `model.bias`;
- `yhat_lin = model(X)`;
- `manual_lin = X @ model.weight.T + model.bias`;
- `same`: `True` nếu hai kết quả trùng nhau.

Câu hỏi: vì sao `weight` có shape `(1, 2)` mà không phải `(2, 1)` như `w` ở bài 1?

In [ ]:
torch.manual_seed(1)
model = nn.Linear(2, 1)
w_shape = model.weight.shape
b_shape = model.bias.shape
yhat_lin = model(X)
manual_lin = X @ model.weight.T + model.bias
same = torch.allclose(yhat_lin, manual_lin)
print(w_shape, b_shape, yhat_lin.shape, same)

In [ ]:
assert tuple(w_shape) == (1, 2) and tuple(b_shape) == (1,)
assert tuple(yhat_lin.shape) == (3, 1)
assert same is True
print("✅ Bài 2 đúng: weight có shape (out_features, in_features)")

## Bài 3. A · Mô hình tự viết cho nhiều đầu vào

Viết lớp `linear_regression(nn.Module)` (như các module trước: `self.linear = nn.Linear(input_size, output_size)`, `forward` gọi lớp đó).
Tạo `model_c = linear_regression(2, 1)`; lưu `keys = list(model_c.state_dict().keys())` và `yhat_c = model_c(X)`.

In [ ]:
class linear_regression(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.linear(x)

model_c = linear_regression(2, 1)
keys = list(model_c.state_dict().keys())
yhat_c = model_c(X)
print(keys, yhat_c.shape)

In [ ]:
assert keys == ["linear.weight", "linear.bias"]
assert tuple(yhat_c.shape) == (3, 1)
print("✅ Bài 3 đúng")

## Bài 4. A · Luyện tập: 4 đặc trưng, và lỗi shape

`X4 = [[11., 12., 13., 14.], [11., 12., 13., 14.]]` (2 mẫu, 4 đặc trưng).
- `y4`: dự đoán của một mô hình `linear_regression(4, 1)` mới tạo. Lưu `shape_y4`.
- `err_msg`: thử đưa `X4` vào `model_c` (mô hình **2** đặc trưng) trong `try/except RuntimeError` và lưu thông báo lỗi.

In [ ]:
X4 = torch.tensor([[11., 12., 13., 14.], [11., 12., 13., 14.]])
model4 = linear_regression(4, 1)
y4 = model4(X4)
shape_y4 = y4.shape

err_msg = None
try:
    model_c(X4)
except RuntimeError as e:
    err_msg = str(e)
print(shape_y4, err_msg)

In [ ]:
assert tuple(shape_y4) == (2, 1)
assert err_msg is not None and "shapes cannot be multiplied" in err_msg
print("✅ Bài 4 đúng")

## Bài 5. C · Nhiều đầu ra: dự đoán và shape

Tạo `model10 = linear_regression(1, 10)` (1 đặc trưng vào, **10** đầu ra). Lưu:
- `out_single`: shape đầu ra khi đưa `[[1.0]]`;
- `out_batch`: shape đầu ra khi đưa `[[1.0], [1.0], [3.0]]`;
- `w10_shape`, `b10_shape`: shape của `weight` và `bias` trong `model10.linear`.

In [ ]:
model10 = linear_regression(1, 10)
out_single = model10(torch.tensor([[1.0]])).shape
out_batch = model10(torch.tensor([[1.0], [1.0], [3.0]])).shape
w10_shape = model10.linear.weight.shape
b10_shape = model10.linear.bias.shape
print(out_single, out_batch, w10_shape, b10_shape)

In [ ]:
assert tuple(out_single) == (1, 10)
assert tuple(out_batch) == (3, 10)
assert tuple(w10_shape) == (10, 1) and tuple(b10_shape) == (10,)
print("✅ Bài 5 đúng")

## Bài 6. B · Dataset hai đặc trưng `Data2D`

Viết lớp `Data2D(Dataset)`:
- `x`: tensor `(20, 2)` toàn 0, rồi gán **cả hai cột** bằng `torch.arange(-1, 1, 0.1)`;
- `f = x @ [[1.], [1.]] + 1`; `y = f + 0.1 * torch.randn((20, 1))`;
- `__getitem__` trả `(x[i], y[i])`, `__len__` trả 20.

Đặt `torch.manual_seed(1)` rồi tạo `data_set = Data2D()`. Lưu `identical_cols = torch.equal(data_set.x[:, 0], data_set.x[:, 1])`.
Câu hỏi: hệ quả của việc hai cột đặc trưng giống hệt nhau là gì? (Bài 7 sẽ trả lời.)

In [ ]:
class Data2D(Dataset):
    def __init__(self):
        self.x = torch.zeros(20, 2)
        self.x[:, 0] = torch.arange(-1, 1, 0.1)
        self.x[:, 1] = torch.arange(-1, 1, 0.1)
        self.w = torch.tensor([[1.0], [1.0]])
        self.b = 1
        self.f = torch.mm(self.x, self.w) + self.b
        self.y = self.f + 0.1 * torch.randn((self.x.shape[0], 1))
        self.len = self.x.shape[0]

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.len

torch.manual_seed(1)
data_set = Data2D()
identical_cols = torch.equal(data_set.x[:, 0], data_set.x[:, 1])
print(identical_cols)

In [ ]:
assert tuple(data_set.x.shape) == (20, 2) and tuple(data_set.y.shape) == (20, 1)
assert identical_cols is True
assert (data_set.y - data_set.f).abs().max().item() < 0.5
print("✅ Bài 6 đúng")

## Bài 7. B · Huấn luyện mô hình hai đặc trưng

Tạo `model = linear_regression(2, 1)`, `optimizer = optim.SGD(model.parameters(), lr=0.1)`, `criterion = nn.MSELoss()`, `train_loader = DataLoader(data_set, batch_size=2)`.
Huấn luyện **100 epoch**, ghi `loss.item()` của **mỗi iteration** vào list `LOSS`.
Rồi lưu: `train_mse` (MSE trên cả `data_set`, không gradient), `w_sum = model.linear.weight.sum().item()`, `b_learned = model.linear.bias.item()`.

**Đoán trước:** trọng số học được có bằng `[1, 1]` không? Tổng của chúng thì sao?

In [ ]:
model = linear_regression(2, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.MSELoss()
train_loader = DataLoader(data_set, batch_size=2)

LOSS = []
for epoch in range(100):
    for x, y in train_loader:
        loss = criterion(model(x), y)
        LOSS.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

with torch.no_grad():
    train_mse = criterion(model(data_set.x), data_set.y).item()
w_sum = model.linear.weight.sum().item()
b_learned = model.linear.bias.item()
plt.plot(LOSS); plt.xlabel("iteration"); plt.ylabel("cost"); plt.show()
print(train_mse, model.linear.weight.tolist(), w_sum, b_learned)

In [ ]:
assert len(LOSS) == 1000
assert train_mse < 0.02
assert abs(w_sum - 2.0) < 0.2, "chỉ tổng w1 + w2 được xác định, ≈ 2"
assert abs(b_learned - 1.0) < 0.15
print("✅ Bài 7 đúng: dự đoán tốt, nhưng w1, w2 riêng lẻ không phải [1, 1]; chỉ tổng của chúng ≈ 2")

## Bài 8. B · Kiểm chứng: trọng số "thật" cũng cho kết quả như nhau

Tạo `alt = linear_regression(2, 1)` và **đặt tay** `weight = [[1., 1.]]`, `bias = [1.]` (các giá trị thật dùng để sinh dữ liệu), bằng `with torch.no_grad(): alt.linear.weight.copy_(...)`.
Tính `alt_mse` (MSE trên `data_set`). So với `train_mse` của mô hình đã huấn luyện: hai con số xấp xỉ nhau.
Lưu `close = abs(alt_mse - train_mse) < 0.01`.

In [ ]:
alt = linear_regression(2, 1)
with torch.no_grad():
    alt.linear.weight.copy_(torch.tensor([[1., 1.]]))
    alt.linear.bias.copy_(torch.tensor([1.]))
    alt_mse = criterion(alt(data_set.x), data_set.y).item()
close = abs(alt_mse - train_mse) < 0.01
print(alt_mse, train_mse, close)

In [ ]:
assert close is True
print("✅ Bài 8 đúng: có vô số cặp (w1, w2) cho dự đoán gần như nhau")

## Bài 9. B · Luyện tập: batch size 10 và validation

Tạo `model1` (mới, 2 đặc trưng) và huấn luyện với `batch_size=10`, `lr=0.1`, 100 epoch trên `data_set`.
Rồi tạo dữ liệu validation: `torch.manual_seed(2)`, `validation_data = Data2D()`.
Tính `val_mse` và `val_mse1`: MSE trên `validation_data` của `model` (batch 2) và `model1` (batch 10), không gradient.

In [ ]:
model1 = linear_regression(2, 1)
optimizer1 = optim.SGD(model1.parameters(), lr=0.1)
for epoch in range(100):
    for x, y in DataLoader(data_set, batch_size=10):
        loss = criterion(model1(x), y)
        optimizer1.zero_grad()
        loss.backward()
        optimizer1.step()

torch.manual_seed(2)
validation_data = Data2D()
with torch.no_grad():
    val_mse = criterion(model(validation_data.x), validation_data.y).item()
    val_mse1 = criterion(model1(validation_data.x), validation_data.y).item()
print(val_mse, val_mse1)

In [ ]:
assert val_mse < 0.02 and val_mse1 < 0.02, "cả hai đều khái quát tốt trên dữ liệu validation"
print("✅ Bài 9 đúng")

## Bài 10. D · Dataset hai đặc trưng, hai đầu ra

Viết lớp `DataMulti(Dataset)`: `x` giống `Data2D` (20 hàng, hai cột bằng `arange(-1, 1, 0.1)`); `w = [[1., -1.], [1., 3.]]` (shape `(2, 2)`), `b = [[1., -1.]]`; `f = x @ w + b`; `y = f + 0.001 * torch.randn((20, 1))` (nhiễu rất nhỏ).
Đặt `torch.manual_seed(1)`, tạo `multi_data`, lưu `y_shape = multi_data.y.shape`.

In [ ]:
class DataMulti(Dataset):
    def __init__(self):
        self.x = torch.zeros(20, 2)
        self.x[:, 0] = torch.arange(-1, 1, 0.1)
        self.x[:, 1] = torch.arange(-1, 1, 0.1)
        self.w = torch.tensor([[1.0, -1.0], [1.0, 3.0]])
        self.b = torch.tensor([[1.0, -1.0]])
        self.f = torch.mm(self.x, self.w) + self.b
        self.y = self.f + 0.001 * torch.randn((self.x.shape[0], 1))
        self.len = self.x.shape[0]

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.len

torch.manual_seed(1)
multi_data = DataMulti()
y_shape = multi_data.y.shape
print(y_shape)

In [ ]:
assert tuple(y_shape) == (20, 2)
assert tuple(multi_data[0][1].shape) == (2,), "mỗi mẫu có 2 giá trị đích"
print("✅ Bài 10 đúng")

## Bài 11. D · Huấn luyện mô hình nhiều đầu ra

Tạo `model_m = linear_regression(2, 2)` (2 đặc trưng vào, 2 đầu ra), `optim.SGD(lr=0.1)`, `nn.MSELoss()`, `DataLoader(multi_data, batch_size=5)`. Huấn luyện 100 epoch, ghi loss mỗi iteration vào `LOSS_m`.
Lưu: `m_weight_shape` (shape của `weight`), `sums = model_m.linear.weight.sum(dim=1)` (tổng trọng số của **từng đầu ra**), `bias_m = model_m.linear.bias`.
Đoán trước: `sums` và `bias_m` bằng gì? (Gợi ý: đặc trưng trùng nhau.)

In [ ]:
model_m = linear_regression(2, 2)
optimizer_m = optim.SGD(model_m.parameters(), lr=0.1)
criterion_m = nn.MSELoss()
LOSS_m = []
for epoch in range(100):
    for x, y in DataLoader(multi_data, batch_size=5):
        loss = criterion_m(model_m(x), y)
        LOSS_m.append(loss.item())
        optimizer_m.zero_grad()
        loss.backward()
        optimizer_m.step()
m_weight_shape = model_m.linear.weight.shape
sums = model_m.linear.weight.sum(dim=1).detach()
bias_m = model_m.linear.bias.detach()
plt.plot(LOSS_m); plt.yscale("log"); plt.xlabel("iteration"); plt.ylabel("cost"); plt.show()
print(m_weight_shape, sums, bias_m, LOSS_m[-1])

In [ ]:
assert len(LOSS_m) == 400
assert tuple(m_weight_shape) == (2, 2)
assert LOSS_m[-1] < 0.01
assert torch.allclose(sums, torch.tensor([2.0, 2.0]), atol=0.1), "mỗi đầu ra: 1+1 và -1+3, cùng bằng 2"
assert torch.allclose(bias_m, torch.tensor([1.0, -1.0]), atol=0.1)
print("✅ Bài 11 đúng")

## Bài 12. D · Dự đoán bằng mô hình đã học

Dùng `model_m` dự đoán tại `x = [[0.5, 0.5]]` (điểm nằm trên đường chéo, giống dữ liệu huấn luyện). Lưu `pred`.
Tính tay giá trị đúng bằng `w`, `b` thật: `x @ [[1,-1],[1,3]] + [1,-1]`. Lưu `expected`.
Tuỳ chọn: thử `x = [[1.0, 0.0]]` (lệch khỏi đường chéo) và quan sát sai số `off_diag_err`; vì sao mô hình có thể sai ở đây?

In [ ]:
x = torch.tensor([[0.5, 0.5]])
with torch.no_grad():
    pred = model_m(x)
expected = torch.mm(x, torch.tensor([[1.0, -1.0], [1.0, 3.0]])) + torch.tensor([[1.0, -1.0]])

x_off = torch.tensor([[1.0, 0.0]])
with torch.no_grad():
    off_true = torch.mm(x_off, torch.tensor([[1.0, -1.0], [1.0, 3.0]])) + torch.tensor([[1.0, -1.0]])
    off_diag_err = (model_m(x_off) - off_true).abs().max().item()
print(pred, expected, off_diag_err)

In [ ]:
assert torch.allclose(pred, expected, atol=0.1), "trên đường chéo, dự đoán phải đúng"
assert expected.tolist() == [[2.0, 0.0]]
assert isinstance(off_diag_err, float)
print("✅ Bài 12 đúng. Ngoài đường chéo mô hình có thể sai vì chưa từng thấy dữ liệu như vậy.")

## Hết